In [4]:
import numpy as np # Linear Algebra
import matplotlib.pyplot as plt # Plotting
import pinocchio as pin # dynamics/simulation

from pinocchio.robot_wrapper import RobotWrapper
from pinocchio.shortcuts import buildModelsFromUrdf
from pinocchio.visualize import MeshcatVisualizer

VISUALIZER = MeshcatVisualizer

ModuleNotFoundError: No module named 'pinocchio.robot_wrapper'

In [5]:
# Pinocchio related
urdf_file = "/usr/share/utheque/iiwa/iiwa.urdf"
pkgs_dirs = "/usr/share/utheque/iiwa"

# Read URDF model
model, collision_model, visual_model = buildModelsFromUrdf(urdf_file, package_dirs=pkgs_dirs)

# Create a RobotWrapper for visualization purposes
robot = RobotWrapper(model, collision_model, visual_model)

# Extract pinocchio model and ata
model = robot.model
data = robot.data

NameError: name 'buildModelsFromUrdf' is not defined

In [6]:
# Visualizer!
robot.setVisualizer(VISUALIZER())
robot.initViewer()
robot.loadViewerModel("pinocchio")

NameError: name 'robot' is not defined

In [7]:
%%html
<iframe src="http://127.0.0.1:7000/static/" width="640" height="480"></iframe>

In [8]:
# function to compute FK for all frames/joints
def fk_all(model, data, q, v = None):
    if v is not None:
        pin.forwardKinematics(model, data, q, v) # FK and Forward Velocities
    else:
        pin.forwardKinematics(model, data, q) # FK
    pin.updateFramePlacements(model, data) # Update frames

In [9]:
def step(model, q, v, dt):
    return pin.integrate(model, q, v * dt)

In [10]:
# Let's simulate!
init_q = pin.randomConfiguration(model)

# end-effector frame
frame_id = model.getFrameId("iiwa_link_ee")

# Display robot
robot.display(init_q)

# timestep
dt = 0.01

AttributeError: module 'pinocchio' has no attribute 'randomConfiguration'

In [11]:
# Let's generate a target
q_rand = pin.randomConfiguration(model)
fk_all(model, data, q_rand)

T_wt = data.oMf[frame_id].copy()

# PI-controller
Kp = 20.
Ki = 0.1

integral = 0.

# Reset robot
q = init_q

# Display robot
robot.display(init_q)

AttributeError: module 'pinocchio' has no attribute 'randomConfiguration'

In [12]:
Nsteps = 100

for i in range(Nsteps):
    # Forward kinematics
    fk_all(model, data, q)
    
    # Jacobians
    pin.computeJointJacobians(model, data, q)
    
    # Get Jacobian of end-effector
    J = pin.getFrameJacobian(model, data, frame_id, pin.ReferenceFrame.LOCAL)
    
    # Get current end-effector pose
    T_wc = data.oMf[frame_id].copy()
    
    # Compute error
    error = pin.log(T_wc.actInv(T_wt)).vector
    
    # Get control signal
    integral += error * dt
    ctrl = Kp * error + Ki * integral
    
    # Get dq via Jacobian pseudoinverse
    v = np.linalg.pinv(J) @ ctrl
    
    # Keep velocity inside limits
    v = np.maximum(np.minimum(v, model.velocityLimit), -model.velocityLimit)
    
    # step simulation
    q = step(model, q, v, dt)
    
    # display
    robot.display(q)

NameError: name 'model' is not defined

In [13]:
%%html
<iframe src="http://127.0.0.1:7000/static/" width="640" height="480"></iframe>